# American Option Pricing Using the Functional Tensor Train (fTT) Approximation

## Libraries:

In [ ]:
from data_setup import *

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
tf.get_logger().setLevel("ERROR")  # silences the tf.compat.v1 deprecations

import time
import keras
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# Libraries Created by the Authors
from nnu import gss_kernels as ssk
from nnu import ftt_regression as ftt
from nnu import points_generator as pgen

# Global
np.set_printoptions(precision = 5, suppress = True)
keras.config.set_floatx("float64")

### Fitting the Functional Tensor Train (ftt) Approximation Using the Alternating Least Squares (ALS)

#### **1.** Preparing the Learning Set

In [ ]:
# Setting a Seed
input_seed = 1

# Setting the Number of Dimensions
ndim = 6

# Setting the Simulation Range
sim_range = 4

# Defining the Number of Points
nx = train.shape[0]

xs = scaler.transform(train.to_numpy()[:, :-1])
ys = train.to_numpy()[:,-1]

#### **2.** Creating (1D) Nodes, Same for Each Dimension

In [ ]:
# Calculating the Number of Nodes
nnodes = 2 * int( pow(nx, 1.0/ndim) )

# Specifying the Kernel Function
kernel = 'invquad' # invquad : inverse quadratic kernel

# Setting the Multiplier for the Scale of the Kernel
scale_mult = 4.0

# Setting a Stretching Factor
stretch = 1.1

# Calculating the Global Scale
global_scale = 2 * sim_range * stretch / nnodes

# Getting the Specified Kernel Function
knl_f = ssk.global_kernel_dict(global_scale * scale_mult)[kernel]

# Generate an Array of Nodes
nodes = np.linspace(-sim_range * stretch, sim_range * stretch, nnodes, endpoint = True)

#### **3.** Specifying the Tensor Train (TT) Ranks

In [ ]:
# Specifying the Rank for the Tensor Train (TT) Decommposition
# For the First and Last Dimensions
tt_flat_rank = 3

# Constructing the List of the Tensor Train Ranks
tt_ranks = [1] + [tt_flat_rank] * (ndim - 1) + [1]

#### **4.** Performing the Fitting via Alternating Least Squares (ALS)

In [ ]:
# Specifying the Number of Iterations for ALS
n_iter = 5

# Recording the Starting Time to Measure the Elapsed Time
start_time = time.time()

# Calculate the Basis Function Values
bf_vals = ftt.basis_function_values(xs, nodes, knl_f)

# Setting a Seed for `np.random`
np.random.seed(input_seed)

# Initialize Tensor Train (TT) Cores for the Decomposition
init_val = None # Setting Initial Values as `None`
tt_cores = ftt.init_tt(tt_ranks, nnodes, init_val = init_val)

# Iterating through the Specified Number of Iterations for ALS
for iter in range(n_iter):
  # Iterating through Each Dimension
  for d in range(ndim):
    # Solve for Current Dimension
    # Updating Tensor Train Cores & Obtaining Fitted Values
    tt_cores, ys_fit = ftt.solve_for_dimension(d = d,
                                               tt_cores = tt_cores,
                                               bf_vals = bf_vals,
                                               ys = ys)
  # Calculate and Print the R2 for the Current Iteration
  r2 = 1 - np.linalg.norm(ys_fit - ys) / np.linalg.norm(ys) # Calculate the R2
  print(f"iter = {iter} r2 = {r2 : .4f}")

iter = 0 r2 =  0.9184
iter = 1 r2 =  0.9755
iter = 2 r2 =  0.9793
iter = 3 r2 =  0.9817
iter = 4 r2 =  0.9834
